# Fine Tuning von Oliver Guhrs German Sentiment

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import pipeline
import pandas as pd
import re
import logging
from tqdm import tqdm
from datetime import datetime
import os

device = 'cuda'
seed_value = 42
senti_df = pd.read_csv('../_data/interruptions_labeled.csv')[['comment_text', 'sentiment']].rename(columns={'comment_text':'text', 'sentiment':'label'})
len(senti_df)
senti_df.head()

,text,label
0,Weiß das das Jugendamt?,1
1,Wieso?,0
2,Erasmus,0
3,Sehr richtig!,2
4,Sie aber auch nicht!,1


In [11]:
model_name_guhr = "oliverguhr/german-sentiment-bert"
model_finetuning = AutoModelForSequenceClassification.from_pretrained(model_name_guhr)
tokenizer_finetuning = AutoTokenizer.from_pretrained(model_name_guhr)

Wie oben festgestellt, folgt das Modell dem Schema {0: 'positive', 1: 'negative', 2: 'neutral'}  
Wir brauchen jetzt also noch eine Abbildung, die unsere gelabelten Daten in dieses Format umwandelt.

In [5]:
id2label_guhr= {0: 'positive', 1: 'negative', 2: 'neutral'}
label2id_guhr = {v: k for k, v in id2label_guhr.items()}
convert_labels_guhr = {0:2, 1:1, 2:0}

In [6]:
senti_df['label'] = senti_df['label'].map(convert_labels_guhr)
senti_df.head()

,text,label
0,Weiß das das Jugendamt?,1
1,Wieso?,2
2,Erasmus,2
3,Sehr richtig!,0
4,Sie aber auch nicht!,1


In [6]:
clean_chars = re.compile(r'[^A-Za-züöäÖÜÄß ]', re.MULTILINE)
clean_http_urls = re.compile(r'https*\S+', re.MULTILINE)
clean_at_mentions = re.compile(r'@\S+', re.MULTILINE)

def replace_numbers(text):
        return text.replace("0"," null").replace("1"," eins").replace("2"," zwei")\
            .replace("3"," drei").replace("4"," vier").replace("5"," fünf") \
            .replace("6"," sechs").replace("7"," sieben").replace("8"," acht") \
            .replace("9"," neun")         

def clean_text(text):
        text = text.replace("\n", " ")        
        text = clean_http_urls.sub('',text)
        text = clean_at_mentions.sub('',text)        
        text = replace_numbers(text)                
        text = clean_chars.sub('', text) # use only text chars                          
        text = ' '.join(text.split()) # substitute multiple whitespace with single whitespace   
        text = text.strip().lower()
        return text

In [12]:
max_length = tokenizer_finetuning.model_max_length
def tokenize_function(data):
    texts = [clean_text(text) for text in data['text']]
    encoded = tokenizer_finetuning.batch_encode_plus(
        texts,
        #padding=True,
        padding = 'max_length',
        add_special_tokens=True,
        max_length = max_length,
        truncation=True,
        return_tensors="pt"
    )
    return encoded



Map:   0%|          | 0/3600 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [8]:
train_df = senti_df.sample(frac=0.80, random_state=42)
test_df = senti_df.drop(train_df.index)  

# use small part of data for proof of concept and lower training times during trial and error
train_dataset = Dataset.from_pandas(train_df[:500], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[:100], preserve_index=False)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

tokenized_dataset = dataset_dict.map(tokenize_function, batched=True)

{'text': 'Eine Straße ist nach ihm benannt! Hier um die Ecke! Das tut deinem Hirn gut, wenn da ein bisschen Luft reinkommt, und deinem Bauch auch!', 'label': 1}


In [9]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42)
eval_dataset = tokenized_dataset["test"].shuffle(seed=42)
print(len(train_dataset))
print(len(eval_dataset))

500
100


In [ ]:

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
def compute_metrics(p):    
    pred, labels = p
    pred = np.argmax(pred, axis=1)
    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    recall = recall_score(y_true=labels, y_pred=pred, average='weighted')
    precision = precision_score(y_true=labels, y_pred=pred, average='weighted')
    f1 = f1_score(y_true=labels, y_pred=pred, average='weighted')

    # compute class-specific precision only for the classes that matter to us
    # those are positive and negative. 
    prec_pos = precision_score(y_true=labels, y_pred=pred, labels=[0], average=None)
    prec_neg = precision_score(y_true=labels, y_pred=pred, labels=[1], average=None)

    mean_prec = ((prec_pos + prec_neg) / 2)[0]
    # using weighted to take class imbalance into account
    return {"accuracy": accuracy, "precision": precision, "precision_pos_neg": mean_prec, "recall": recall, "f1": f1}

In [11]:
class LoggingCallback(TrainerCallback):

    def __init__(self, logdir):
        super().__init__()
        self.logfiles_prefix = logdir
        self.log_counter = 0
        self.eval_counter = 0

    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.log_counter == 0:
            with open(f"{self.logfiles_prefix}/train_loss.txt", "w") as f:
                f.write(f"step,train_loss\n")
        
        if logs is not None and "loss" in logs:
            with open(f"{self.logfiles_prefix}/train_loss.txt", "a") as f:
                f.write(f"{state.global_step},{logs['loss']}\n")
        self.log_counter +=1
            
                   
    def on_evaluate(self, args, state, control, **kwargs):
        if self.eval_counter == 0:
            with open(f"{self.logfiles_prefix}/eval_metrics.txt", "w") as f:
                f.write(f"step,eval_loss,eval_accuracy,epoch\n")

        logs = kwargs.get('metrics',{})
        if logs:
            with open(f"{self.logfiles_prefix}/eval_metrics.txt", "a") as f:
                f.write(f"{state.global_step},{logs['eval_loss']},{logs['eval_accuracy']},{logs['epoch']}\n")
        self.eval_counter +=1



In [10]:
# tensorboard_cell
from transformers import TrainingArguments, Trainer, TrainerCallback

train = False
if train:
    model_finetuning.to(device)
    current_datetime = datetime.now()
    formatted_datetime = current_datetime.strftime("%Y-%m-%d_%H-%M-%S") # make it okay to use for filenames 

    output_std = 'logs/standard/' + formatted_datetime+'/output/'
    log_std = 'logs/standard/' + formatted_datetime +'/logs/'
    # Define the training arguments with the seed parameter
    training_args_standard = TrainingArguments(
        output_dir=output_std, 
        evaluation_strategy="steps",
        eval_steps = 30,
        num_train_epochs=4,
        seed=seed_value,  # Set the seed here
        save_strategy="steps",
        save_steps=100,
        logging_dir=log_std,  # Specify the logging directory
        logging_steps=10,  # Log every 10 steps
        logging_strategy='steps', 
        report_to = 'tensorboard'
    )

In [12]:
if train:
    trainer_standard = Trainer(
        model=model_finetuning,
        args=training_args_standard,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[LoggingCallback(output_std)]
    )

In [13]:
if train:
# Train the model
    trainer_standard.train()

### Tensorboard to monitor loss on eval and train dataset during training

In [14]:

#%load_ext tensorboard
%load_ext tensorboard

%tensorboard --logdir ./logs/

Reusing TensorBoard on port 6006 (pid 19148), started 0:16:08 ago. (Use '!kill 19148' to kill it.)

In [15]:
#model_path_standard = "models/guhr_bert_finetuned_standard_4ep/"
#trainer_standard.save_model(model_path_standard)

Der Versuch, mit einem kleinen Teil der Daten erstmal ein Proof of Concept zu generieren, hat die ersten Probleme aufgezeigt.  
Das BERT-Modell overfittet unfassbar schnell auf die Trainingsdaten. Schon nach der zweiten Epoche ist der Loss auf dem Eval-Datensatz merkbar höher, als der auf dem Trainingsdaten.  
Wenn das Modell jetzt im Folgenden mehr Daten trainiert wird, dann verzögert sich das Problem, es wird aber vermutlich wieder auftreten.  
Deswegen habe ich mich dafür entschieden Early Stopping einzubauen, um das finale Modell zu trainieren.

In [2]:
import pandas as pd

In [1]:
import plotly.graph_objects as go


def plot_train_eval_loss(path_to_output_dir):
    fig = go.Figure()
    train_loss = pd.read_csv(path_to_output_dir + 'train_loss.txt' )
    eval_metrics = pd.read_csv(path_to_output_dir +'eval_metrics.txt')
    # add evaluation loss line in red
    fig.add_trace(go.Scatter(x=eval_metrics['step'], y=eval_metrics['eval_loss'],
                            mode='lines+markers', name='Evaluation Loss',
                            line=dict(color='red')))

    # add training loss line in blue
    fig.add_trace(go.Scatter(x=train_loss['step'], y=train_loss['train_loss'],
                            mode='lines+markers', name='Training Loss',
                            line=dict(color='blue')))

    fig.update_layout(
        title='Training and Evaluation Loss over Steps',
        xaxis_title='Step',
        yaxis_title='Loss',
        legend=dict(x=0.1, y=1.1),
        hovermode='x unified'
    )

    fig.show()

plot_train_eval_loss('logs/standard/2024-06-21_17-19-06/output/')

NameError: name 'pd' is not defined

# Finetuning mit Early Stopping

In [9]:
train_dataset = Dataset.from_pandas(train_df, preserve_index=False) # the whole dataset we have labeled
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

tokenized_dataset = dataset_dict.map(tokenize_function, batched=True)

train_dataset = tokenized_dataset["train"].shuffle(seed=42)
eval_dataset = tokenized_dataset["test"].shuffle(seed=42)
print(len(train_dataset))
print(len(eval_dataset))

NameError: name 'tokenize_function' is not defined

In [18]:

current_datetime = datetime.now()
formatted_datetime = current_datetime.strftime("%Y-%m-%d_%H-%M-%S") # make it okay to use for filenames 

output_finetuning = 'logs/earlyst/' + formatted_datetime+'/output/'
log_finetuning = 'logs/earlyst/' + formatted_datetime +'/logs/'
training_args_standard = TrainingArguments(
    output_dir=output_finetuning, 
    evaluation_strategy="steps",
    eval_steps = 30,
    num_train_epochs=8,
    seed=seed_value, 
    save_strategy="steps",
    save_steps=30,
    logging_dir=log_finetuning,  
    logging_steps=10,  # log every 10 steps
    logging_strategy='steps', 
    report_to = 'tensorboard',
    save_total_limit = 10,
    metric_for_best_model = 'eval_loss',
    greater_is_better=False, 
    load_best_model_at_end=True,
    per_device_eval_batch_size= 24,
    per_device_train_batch_size= 24
)


c:\ProgFiles\PythonVenvs\.NLPvenv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [19]:
from transformers import EarlyStoppingCallback
trainer_finetuning = Trainer(
    model=model_finetuning,
    args=training_args_standard,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=5), LoggingCallback(output_finetuning)]
)

In [20]:
train = True
if train:
    trainer_finetuning.train()  

  0%|          | 0/1200 [00:00<?, ?it/s]

c:\ProgFiles\PythonVenvs\.NLPvenv\Lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'loss': 1.2514, 'grad_norm': 13.467867851257324, 'learning_rate': 4.958333333333334e-05, 'epoch': 0.07}
{'loss': 0.6346, 'grad_norm': 5.308777332305908, 'learning_rate': 4.9166666666666665e-05, 'epoch': 0.13}
{'loss': 0.6923, 'grad_norm': 8.498133659362793, 'learning_rate': 4.875e-05, 'epoch': 0.2}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.7369009256362915, 'eval_accuracy': 0.7155555555555555, 'eval_precision': 0.7227162728285514, 'eval_precision_pos_neg': 0.6997269116186693, 'eval_recall': 0.7155555555555555, 'eval_f1': 0.7010961344873065, 'eval_runtime': 22.6309, 'eval_samples_per_second': 39.769, 'eval_steps_per_second': 1.679, 'epoch': 0.2}
{'loss': 0.6022, 'grad_norm': 4.00246000289917, 'learning_rate': 4.8333333333333334e-05, 'epoch': 0.27}
{'loss': 0.6042, 'grad_norm': 3.727978229522705, 'learning_rate': 4.791666666666667e-05, 'epoch': 0.33}
{'loss': 0.6248, 'grad_norm': 5.448042869567871, 'learning_rate': 4.75e-05, 'epoch': 0.4}


...

{'eval_loss': 0.6477572917938232, 'eval_accuracy': 0.7577777777777778, 'eval_precision': 0.755257539367706, 'eval_precision_pos_neg': 0.7379551820728292, 'eval_recall': 0.7577777777777778, 'eval_f1': 0.7558804341770502, 'eval_runtime': 20.5977, 'eval_samples_per_second': 43.694, 'eval_steps_per_second': 1.845, 'epoch': 1.6}
{'loss': 0.3454, 'grad_norm': 7.734441757202148, 'learning_rate': 3.958333333333333e-05, 'epoch': 1.67}
{'loss': 0.4175, 'grad_norm': 12.098886489868164, 'learning_rate': 3.9166666666666665e-05, 'epoch': 1.73}
{'loss': 0.4269, 'grad_norm': 8.133739471435547, 'learning_rate': 3.875e-05, 'epoch': 1.8}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.665368378162384, 'eval_accuracy': 0.7533333333333333, 'eval_precision': 0.7521406849358555, 'eval_precision_pos_neg': 0.767720717408179, 'eval_recall': 0.7533333333333333, 'eval_f1': 0.7450307636859944, 'eval_runtime': 20.6149, 'eval_samples_per_second': 43.658, 'eval_steps_per_second': 1.843, 'epoch': 1.8}
{'loss': 0.3792, 'grad_norm': 4.489762306213379, 'learning_rate': 3.8333333333333334e-05, 'epoch': 1.87}
{'loss': 0.4208, 'grad_norm': 7.393604278564453, 'learning_rate': 3.791666666666667e-05, 'epoch': 1.93}
{'loss': 0.4635, 'grad_norm': 7.535828113555908, 'learning_rate': 3.7500000000000003e-05, 'epoch': 2.0}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.6431262493133545, 'eval_accuracy': 0.7633333333333333, 'eval_precision': 0.7613544857189545, 'eval_precision_pos_neg': 0.7757168251024036, 'eval_recall': 0.7633333333333333, 'eval_f1': 0.758898303516164, 'eval_runtime': 20.6082, 'eval_samples_per_second': 43.672, 'eval_steps_per_second': 1.844, 'epoch': 2.0}
{'train_runtime': 4189.0494, 'train_samples_per_second': 6.875, 'train_steps_per_second': 0.286, 'train_loss': 0.5293156901995341, 'epoch': 2.0}


Ich habe einige unterschiedliche Trainingsparameter ausprobiert. 
Standard mit Batch-Größe 8, early stopping auf eval_accuracy, eval loss
Batch-Größe 16
Batch Größe 32 -> ging auf meiner Hardware nicht

In [21]:
trainer_finetuning.save_model('./models/guhr_bert_finetuned_early_stop_eval_loss_b24')  

In [ ]:
trainer_finetuning.save_model('./models/guhr_bert_finetuned_weighted_cross_entropy')

In [22]:
err

NameError: name 'err' is not defined

In [ ]:
trainer_finetuning.save_model('./models/guhr_bert_finetuned_weighted_cross_entropy')

In [13]:
plot_train_eval_loss('logs/finetuning/2024-06-17_12-21-42/output/')

# Vor + Nachteile bei Scores irgendwie beschreiben

# Bemerkung: zu lange Trainingszeit:

Übertraining schon ab Schritt 200-300. 

Ich habe erst mit 3000 gelabelten Daten trainiert, dann mit 4500. Der Unterschied in der Accuracy liegt bei nur einem Prozent.  
Wenn man die 

# Andere Loss-Funktionen

In [27]:
# Optional if (re)starting the Notebook from here:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback, AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import pipeline
import pandas as pd
import re
import logging
from tqdm import tqdm
from datetime import datetime
import os
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

device = 'cuda'
seed_value = 42
senti_df = pd.read_csv('../_data/interruptions_labeled.csv')[['comment_text', 'sentiment']].rename(columns={'comment_text':'text', 'sentiment':'label'})
len(senti_df)
senti_df.head()

,text,label
0,Weiß das das Jugendamt?,1
1,Wieso?,0
2,Erasmus,0
3,Sehr richtig!,2
4,Sie aber auch nicht!,1


In [28]:
model_name_guhr = "oliverguhr/german-sentiment-bert"
model_finetuning = AutoModelForSequenceClassification.from_pretrained(model_name_guhr)
tokenizer_finetuning = AutoTokenizer.from_pretrained(model_name_guhr)

In [29]:
id2label_guhr= {0: 'positive', 1: 'negative', 2: 'neutral'}
label2id_guhr = {v: k for k, v in id2label_guhr.items()}
convert_labels_guhr = {0:2, 1:1, 2:0}

In [30]:
senti_df['label'] = senti_df['label'].map(convert_labels_guhr)
senti_df.head()

,text,label
0,Weiß das das Jugendamt?,1
1,Wieso?,2
2,Erasmus,2
3,Sehr richtig!,0
4,Sie aber auch nicht!,1


In [31]:
clean_chars = re.compile(r'[^A-Za-züöäÖÜÄß ]', re.MULTILINE)
clean_http_urls = re.compile(r'https*\S+', re.MULTILINE)
clean_at_mentions = re.compile(r'@\S+', re.MULTILINE)

def replace_numbers(text):
        return text.replace("0"," null").replace("1"," eins").replace("2"," zwei")\
            .replace("3"," drei").replace("4"," vier").replace("5"," fünf") \
            .replace("6"," sechs").replace("7"," sieben").replace("8"," acht") \
            .replace("9"," neun")         

def clean_text(text):
        text = text.replace("\n", " ")        
        text = clean_http_urls.sub('',text)
        text = clean_at_mentions.sub('',text)        
        text = replace_numbers(text)                
        text = clean_chars.sub('', text) # use only text chars                          
        text = ' '.join(text.split()) # substitute multiple whitespace with single whitespace   
        text = text.strip().lower()
        return text

In [32]:
max_length = tokenizer_finetuning.model_max_length
def tokenize_function(data):
    texts = [clean_text(text) for text in data['text']]
    encoded = tokenizer_finetuning.batch_encode_plus(
        texts,
        #padding=True,
        padding = 'max_length',
        add_special_tokens=True,
        max_length = max_length,
        truncation=True,
        return_tensors="pt"
    )
    return encoded

In [33]:
train_df = senti_df.sample(frac=0.80, random_state=42)
test_df = senti_df.drop(train_df.index)  

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

tokenized_dataset = dataset_dict.map(tokenize_function, batched=True)

train_dataset = tokenized_dataset["train"].shuffle(seed=42)
eval_dataset = tokenized_dataset["test"].shuffle(seed=42)
print(len(train_dataset))
print(len(eval_dataset))

Map:   0%|          | 0/3600 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

3600
900


In [34]:
def focal_loss(labels, logits, alpha=1, gamma=2):
    ce_loss = torch.nn.functional.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce_loss)
    focal_loss = alpha * ((1 - pt) ** gamma) * ce_loss
    return focal_loss.mean().item()

def compute_metrics(p):
    pred_logits, labels = p
    pred_probs = torch.softmax(torch.tensor(pred_logits), dim=1)
    pred = np.argmax(pred_probs.numpy(), axis=1)

    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    recall = recall_score(y_true=labels, y_pred=pred, average='weighted')
    precision = precision_score(y_true=labels, y_pred=pred, average='weighted')
    f1 = f1_score(y_true=labels, y_pred=pred, average='weighted')

    # compute class-specific precision only for the classes that matter to us
    # those are positive and negative. 
    prec_pos = precision_score(y_true=labels, y_pred=pred, labels=[0], average=None)
    prec_neg = precision_score(y_true=labels, y_pred=pred, labels=[1], average=None)

    mean_prec = ((prec_pos + prec_neg) / 2)[0]

    return {
        "accuracy": accuracy,
        "precision": precision,
        "precision_pos_neg": mean_prec,
        "recall": recall,
        "f1": f1
    }

In [35]:
class LoggingCallback(TrainerCallback):

    def __init__(self, logdir):
        super().__init__()
        self.logfiles_prefix = logdir
        self.log_counter = 0
        self.eval_counter = 0

    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.log_counter == 0:
            with open(f"{self.logfiles_prefix}/train_loss.txt", "w") as f:
                f.write(f"step,train_loss\n")
        
        if logs is not None and "loss" in logs:
            with open(f"{self.logfiles_prefix}/train_loss.txt", "a") as f:
                f.write(f"{state.global_step},{logs['loss']}\n")
        self.log_counter +=1
            
                   
    def on_evaluate(self, args, state, control, **kwargs):
        if self.eval_counter == 0:
            with open(f"{self.logfiles_prefix}/eval_metrics.txt", "w") as f:
                f.write(f"step,eval_loss,eval_accuracy,epoch\n")

        logs = kwargs.get('metrics',{})
        if logs:
            with open(f"{self.logfiles_prefix}/eval_metrics.txt", "a") as f:
                f.write(f"{state.global_step},{logs['eval_loss']},{logs['eval_accuracy']},{logs['epoch']}\n")
        self.eval_counter +=1



In [11]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = nn.CrossEntropyLoss()(inputs, targets)
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

In [12]:
class FocalLossTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = FocalLoss()

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = self.criterion(logits, labels)
        return (loss, outputs) if return_outputs else loss
    
    def evaluation_step(self, model, inputs):
        model.eval()
        with torch.no_grad():
            loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
        return loss.item(), outputs


In [13]:
current_datetime = datetime.now()
formatted_datetime = current_datetime.strftime("%Y-%m-%d_%H-%M-%S") # make it okay to use for filenames 

output_finetuning = 'logs/focal/' + formatted_datetime+'/output/'
log_finetuning = 'logs/focal/' + formatted_datetime +'/logs/'


training_args_focal = TrainingArguments(
    output_dir=output_finetuning, 
    evaluation_strategy="steps",
    eval_steps = 20,
    num_train_epochs=8,
    seed=seed_value, 
    save_strategy="steps",
    save_steps=20,
    logging_dir=log_finetuning,  
    logging_steps=10,  # log every 10 steps
    logging_strategy='steps', 
    report_to = 'tensorboard',
    save_total_limit = 10,
    metric_for_best_model = 'eval_loss',
    greater_is_better=False, 
    load_best_model_at_end=True,
    per_device_eval_batch_size= 24,
    per_device_train_batch_size= 24
)



c:\ProgFiles\PythonVenvs\.NLPvenv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning:

`evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead



In [14]:
from transformers import EarlyStoppingCallback
trainer_finetuning = FocalLossTrainer(
    model=model_finetuning,
    args=training_args_focal,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=5), LoggingCallback(output_finetuning)]
)

In [15]:
train = True
if train:
    trainer_finetuning.train()  

  0%|          | 0/1200 [00:00<?, ?it/s]

c:\ProgFiles\PythonVenvs\.NLPvenv\Lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning:

1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)



{'loss': 0.6841, 'grad_norm': 8.75127124786377, 'learning_rate': 4.958333333333334e-05, 'epoch': 0.07}
{'loss': 0.1454, 'grad_norm': 2.2241227626800537, 'learning_rate': 4.9166666666666665e-05, 'epoch': 0.13}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.20665708184242249, 'eval_accuracy': 0.6966666666666667, 'eval_precision': 0.6909076161453842, 'eval_precision_pos_neg': 0.6742927221211993, 'eval_recall': 0.6966666666666667, 'eval_f1': 0.6894533103646283, 'eval_runtime': 20.6953, 'eval_samples_per_second': 43.488, 'eval_steps_per_second': 1.836, 'epoch': 0.13}
{'loss': 0.1868, 'grad_norm': 3.59041166305542, 'learning_rate': 4.875e-05, 'epoch': 0.2}
{'loss': 0.1367, 'grad_norm': 1.907149314880371, 'learning_rate': 4.8333333333333334e-05, 'epoch': 0.27}


...

{'eval_loss': 0.1463586837053299, 'eval_accuracy': 0.7633333333333333, 'eval_precision': 0.781577974976786, 'eval_precision_pos_neg': 0.8348618527977711, 'eval_recall': 0.7633333333333333, 'eval_f1': 0.7463369427372939, 'eval_runtime': 20.5324, 'eval_samples_per_second': 43.833, 'eval_steps_per_second': 1.851, 'epoch': 1.33}
{'loss': 0.0442, 'grad_norm': 3.3784286975860596, 'learning_rate': 4.125e-05, 'epoch': 1.4}
{'loss': 0.0615, 'grad_norm': 2.287179708480835, 'learning_rate': 4.0833333333333334e-05, 'epoch': 1.47}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.13981497287750244, 'eval_accuracy': 0.7677777777777778, 'eval_precision': 0.7672242532622566, 'eval_precision_pos_neg': 0.7955164466792374, 'eval_recall': 0.7677777777777778, 'eval_f1': 0.761992643818536, 'eval_runtime': 20.5225, 'eval_samples_per_second': 43.854, 'eval_steps_per_second': 1.852, 'epoch': 1.47}
{'loss': 0.0536, 'grad_norm': 1.7137349843978882, 'learning_rate': 4.041666666666667e-05, 'epoch': 1.53}
{'loss': 0.0649, 'grad_norm': 4.141526699066162, 'learning_rate': 4e-05, 'epoch': 1.6}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.17352887988090515, 'eval_accuracy': 0.7511111111111111, 'eval_precision': 0.7502476295550399, 'eval_precision_pos_neg': 0.7275018315018316, 'eval_recall': 0.7511111111111111, 'eval_f1': 0.7505661881147135, 'eval_runtime': 20.5357, 'eval_samples_per_second': 43.826, 'eval_steps_per_second': 1.85, 'epoch': 1.6}
{'train_runtime': 3327.7562, 'train_samples_per_second': 8.654, 'train_steps_per_second': 0.361, 'train_loss': 0.12953142672777176, 'epoch': 1.6}


In [16]:
trainer_finetuning.save_model('./models/focal_neu_4_b24')  

In [17]:
plot_train_eval_loss('logs/focal/2024-06-22_23-25-12/output/') # mit batch Größe 8

In [18]:
plot_train_eval_loss('logs/focal/2024-06-22_18-29-22/output/')

In [19]:
import torch 
torch.cuda.empty_cache()

## Weighted Cross Entropy Loss

In [38]:
from collections import Counter
class_counts = Counter(train_df['label'])
total_samples = len(train_df)
class_weights = {label: total_samples / count for label, count in class_counts.items()}

class_weights

{2: 1.765571358509073, 0: 7.67590618336887, 1: 3.2967032967032965}

In [51]:
from sklearn.utils.class_weight import compute_class_weight
labels = senti_df['label'].to_numpy()
classes = np.unique(labels)


class_weights= torch.tensor(compute_class_weight(class_weight = 'balanced', classes = classes,y = labels), dtype=torch.float)
criterion = nn.CrossEntropyLoss(weight=class_weights,reduction='mean')

In [52]:
class WeightedCETrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = weightedCEcriterion

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = self.criterion(logits, labels)
        return (loss, outputs) if return_outputs else loss
    
    def evaluation_step(self, model, inputs):
        model.eval()
        with torch.no_grad():
            loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
        return loss.item(), outputs


In [53]:
current_datetime = datetime.now()
formatted_datetime = current_datetime.strftime("%Y-%m-%d_%H-%M-%S") # make it okay to use for filenames 

output_finetuning = 'logs/wCE/' + formatted_datetime+'/output/'
log_finetuning = 'logs/wCE/' + formatted_datetime +'/logs/'

training_args_weighted_ce = TrainingArguments(
    output_dir=output_finetuning, 
    evaluation_strategy="steps",
    eval_steps = 20,
    num_train_epochs=8,
    seed=seed_value, 
    save_strategy="steps",
    save_steps=20,
    logging_dir=log_finetuning,  
    logging_steps=10,  # log every 10 steps
    logging_strategy='steps', 
    report_to = 'tensorboard',
    save_total_limit = 10,
    metric_for_best_model = 'eval_loss',
    greater_is_better=False, 
    load_best_model_at_end=True,
    per_device_eval_batch_size= 24,
    per_device_train_batch_size= 24
)

c:\ProgFiles\PythonVenvs\.NLPvenv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning:

`evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead



In [54]:
trainer_finetuning = WeightedCETrainer(
    model=model_finetuning,
    args=training_args_weighted_ce,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=5), LoggingCallback(output_finetuning)],

)

In [55]:
train = True
if train:
    trainer_finetuning.train()

  0%|          | 0/1200 [00:00<?, ?it/s]

{'loss': 1.3048, 'grad_norm': 8.379194259643555, 'learning_rate': 4.958333333333334e-05, 'epoch': 0.07}
{'loss': 0.5963, 'grad_norm': 5.707910537719727, 'learning_rate': 4.9166666666666665e-05, 'epoch': 0.13}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.7125074863433838, 'eval_accuracy': 0.68, 'eval_precision': 0.682686603788053, 'eval_precision_pos_neg': 0.6055555555555556, 'eval_recall': 0.68, 'eval_f1': 0.6784683254161796, 'eval_runtime': 318.5765, 'eval_samples_per_second': 2.825, 'eval_steps_per_second': 0.119, 'epoch': 0.13}
{'loss': 0.7733, 'grad_norm': 13.956857681274414, 'learning_rate': 4.875e-05, 'epoch': 0.2}
{'loss': 0.605, 'grad_norm': 6.725024700164795, 'learning_rate': 4.8333333333333334e-05, 'epoch': 0.27}


  0%|          | 0/38 [00:00<?, ?it/s]

...

{'eval_loss': 0.724410891532898, 'eval_accuracy': 0.7488888888888889, 'eval_precision': 0.7493724434472098, 'eval_precision_pos_neg': 0.757179269328802, 'eval_recall': 0.7488888888888889, 'eval_f1': 0.7487682691468641, 'eval_runtime': 414.1249, 'eval_samples_per_second': 2.173, 'eval_steps_per_second': 0.092, 'epoch': 1.47}
{'loss': 0.4428, 'grad_norm': 11.615581512451172, 'learning_rate': 4.041666666666667e-05, 'epoch': 1.53}
{'loss': 0.4267, 'grad_norm': 10.169219970703125, 'learning_rate': 4e-05, 'epoch': 1.6}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.658751368522644, 'eval_accuracy': 0.6755555555555556, 'eval_precision': 0.7087820986769072, 'eval_precision_pos_neg': 0.6109649122807017, 'eval_recall': 0.6755555555555556, 'eval_f1': 0.67622568810238, 'eval_runtime': 414.1494, 'eval_samples_per_second': 2.173, 'eval_steps_per_second': 0.092, 'epoch': 1.6}
{'loss': 0.3524, 'grad_norm': 8.047410011291504, 'learning_rate': 3.958333333333333e-05, 'epoch': 1.67}
{'loss': 0.4414, 'grad_norm': 8.283544540405273, 'learning_rate': 3.9166666666666665e-05, 'epoch': 1.73}


  0%|          | 0/38 [00:00<?, ?it/s]

{'eval_loss': 0.6471179127693176, 'eval_accuracy': 0.7433333333333333, 'eval_precision': 0.7422674520799761, 'eval_precision_pos_neg': 0.7255798660927786, 'eval_recall': 0.7433333333333333, 'eval_f1': 0.7427204289857726, 'eval_runtime': 434.3661, 'eval_samples_per_second': 2.072, 'eval_steps_per_second': 0.087, 'epoch': 1.73}
{'train_runtime': 11630.4602, 'train_samples_per_second': 2.476, 'train_steps_per_second': 0.103, 'train_loss': 0.5586248223598187, 'epoch': 1.73}


In [56]:
trainer_finetuning.save_model('./models/wce_1_b24_x')

In [58]:
plot_train_eval_loss('logs/wCE/2024-06-23_08-54-42/output/')